In [1]:
!pip install sqlalchemy psycopg2-binary pandas


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sqlalchemy import create_engine
username='postgres'
password='shree'
port=5432
database_name='globalsesmic_db'
host_name='localhost'
engine=create_engine(f'postgresql+psycopg2://{username}:{password}@{host_name}:{port}/{database_name}')

In [3]:
import pandas as pd

In [21]:
df=pd.read_csv('earthquake_clean.csv')
print("file read successfully")

file read successfully


In [22]:
df['place']=df['place'].str.strip()
df['place']

0               29 km SW of Villa Basilio Nievas, Argentina
1                                                      Fiji
2                           103 km SW of Basco, Philippines
3                                   114 km N of M?n?b, Iran
4         184 km ENE of Olonkinbyen, Svalbard and Jan Mayen
                                ...                        
113526                                   Izu Islands, Japan
113527                                   Izu Islands, Japan
113528                  off the east coast of Honshu, Japan
113529     63 km N of Charlotte Amalie, U.S. Virgin Islands
113530                                     Macquarie Island
Name: place, Length: 113531, dtype: object

In [23]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
latitude,113531.0,12.096730,32.027786,-84.4932,-14.7137,14.9010,39.05455,87.3752
longitude,113531.0,7.457674,129.009475,-179.9997,-113.3237,24.3400,130.55135,179.9994
depth_km,113531.0,71.975188,123.384085,0.0000,10.0000,21.7060,71.23950,683.5780
mag,113531.0,4.257262,0.609661,3.0000,4.0000,4.3000,4.60000,8.8000
tsunami,113531.0,0.005805,0.075967,0.0000,0.0000,0.0000,0.00000,1.0000
felt,113531.0,15.157050,670.678048,0.0000,0.0000,0.0000,0.00000,184666.0000
cdi,113531.0,3.119230,0.591251,0.0000,3.1000,3.1000,3.10000,9.1000
mmi,113531.0,3.400637,0.474372,0.0000,3.4125,3.4125,3.41250,9.9530
sig,113531.0,287.632964,91.418839,138.0000,247.0000,284.0000,326.00000,2910.0000
nst,113531.0,34.327091,39.498222,0.0000,7.0000,24.0000,46.00000,619.0000


In [24]:
df.to_sql("earthquakes",con=engine,if_exists='replace',index=False)

133

In [25]:
sql_query = "SELECT * FROM earthquakes LIMIT 5;"

In [26]:
df_result = pd.read_sql(sql_query, engine)
df_result

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,gap,type,region,country,year,month,day,day_of_week,depth_flag,mag_flag
0,us6000ddi8,2021-01-31 23:20:49.923,2021-04-16 19:02:44.040,-31.7493,-68.9337,17.27,4.7,mwr,"29 km SW of Villa Basilio Nievas, Argentina",reviewed,...,42.0,earthquake,29 km SW of Villa Basilio Nievas,Argentina,2021,January,31,Sunday,Shallow,Strong
1,us6000dev6,2021-01-31 23:08:17.161,2021-04-16 19:03:47.040,-15.4902,-177.2052,426.71,4.1,mb,Fiji,reviewed,...,64.0,earthquake,Fiji,Fiji,2021,January,31,Sunday,Deep,Strong
2,us6000dev5,2021-01-31 22:54:19.760,2021-04-16 19:03:47.040,19.7529,121.3159,46.73,4.7,mb,"103 km SW of Basco, Philippines",reviewed,...,106.0,earthquake,103 km SW of Basco,Philippines,2021,January,31,Sunday,Shallow,Strong
3,us6000ddhs,2021-01-31 22:06:00.832,2021-04-16 19:02:43.040,28.1524,57.2570,10.00,4.9,mb,"114 km N of M?n?b, Iran",reviewed,...,71.0,earthquake,114 km N of M?n?b,Iran,2021,January,31,Sunday,Shallow,Strong
4,us6000dev4,2021-01-31 21:51:14.016,2021-04-16 19:03:46.040,71.3212,-3.7578,10.00,4.0,mb,"184 km ENE of Olonkinbyen, Svalbard and Jan Mayen",reviewed,...,65.0,earthquake,184 km ENE of Olonkinbyen,Svalbard and Jan Mayen,2021,January,31,Sunday,Shallow,Strong


Magnitude & Depth
1. Top 10 strongest earthquakes (mag).

In [27]:
query_1="SELECT mag FROM earthquakes ORDER BY mag ASC LIMIT 10"

In [28]:
df_result1 = pd.read_sql(query_1, engine)
print(df_result1)

   mag
0  3.0
1  3.0
2  3.0
3  3.0
4  3.0
5  3.0
6  3.0
7  3.0
8  3.0
9  3.0


2. Top 10 deepest earthquakes (depth_km).

In [29]:
query_2='''SELECT place,depth_km
            FROM earthquakes
            WHERE mag > 7.0
            ORDER BY depth_km DESC
            LIMIT 10'''

In [30]:
df_result2 = pd.read_sql(query_2, engine)
print(df_result2)

                                      place  depth_km
0         106 km WSW of Sangay, Philippines   639.503
1         57 km NNW of Kota Belud, Malaysia   629.000
2                                   Vanuatu   527.000
3         180 km NNE of Gili Air, Indonesia   500.000
4                 10 km W of Azángaro, Peru   236.000
5                 158 km W of Neiafu, Tonga   234.000
6                82 km WNW of Hihifo, Tonga   210.000
7                 274 km SW of Houma, Tonga   179.000
8       125 km NNE of Lospalos, Timor Leste   165.490
9  41 km ESE of San Pedro de Atacama, Chile   127.291


3. Shallow earthquakes < 50 km and mag > 7.5

In [31]:
query3='''SELECT depth_km,place,mag FROM earthquakes
        Where depth_km <50 AND mag > 7.5
        ORDER BY mag DESC'''

In [32]:
df_result3 = pd.read_sql(query3, engine)
print(df_result3)

    depth_km                                         place  mag
0     35.000              2025 Kamchatka Peninsula, Russia  8.8
1     35.000                          2021 Chignik, Alaska  8.2
2     22.790                   2021 South Sandwich Islands  8.1
3     28.930            2021 Kermadec Islands, New Zealand  8.1
4     27.000  140 km E of Petropavlovsk-Kamchatsky, Russia  7.8
5     10.000            Pazarcik earthquake, Kahramanmaras  7.8
6     10.000              southeast of the Loyalty Islands  7.7
7     10.000                2025 Mandalay, Burma (Myanmar)  7.7
8     18.053              southeast of the Loyalty Islands  7.7
9     40.720                 2025 Aomori Prefecture, Japan  7.6
10    26.943                35 km SSW of Aguililla, Mexico  7.6
11    40.000                 19 km E of Gamut, Philippines  7.6
12    14.326     210 km SSW of George Town, Cayman Islands  7.6
13     5.639                                 Drake Passage  7.6


4. Average depth per continent.

5. Average magnitude per magnitude type (magType).

In [33]:
query5=''' SELECT "magType", AVG(mag) AS avg_magnitude
            FROM earthquakes
            GROUP BY "magType"
            ORDER BY avg_magnitude DESC'''

In [34]:
df_result5 = pd.read_sql(query5, engine)
print(df_result5)

     magType  avg_magnitude
0        mwc       6.150000
1      ms_20       5.800000
2        mwb       5.800000
3        mww       5.368347
4        mwp       5.250000
5      ms_vx       4.600000
6         mb       4.423724
7        mwr       4.328117
8         mh       4.100000
9         mw       3.939507
10       mlr       3.480000
11       mlv       3.469288
12        md       3.441209
13  mltexnet       3.367442
14       mlg       3.300000
15        ml       3.274643
16     mb_lg       3.204032


Time Analysis

6. Year with most earthquakes.

In [35]:
query6=''' SELECT year,COUNT(*) AS total_earthquakes
            FROM earthquakes
            GROUP BY year
            order by year desc'''

In [36]:
# Run this to see all your column names
pd.read_sql("SELECT * FROM earthquakes LIMIT 1", engine).columns.tolist()

['id',
 'time',
 'updated',
 'latitude',
 'longitude',
 'depth_km',
 'mag',
 'magType',
 'place',
 'status',
 'tsunami',
 'alert',
 'felt',
 'cdi',
 'mmi',
 'sig',
 'net',
 'code',
 'ids',
 'sources',
 'types',
 'nst',
 'dmin',
 'rms',
 'gap',
 'type',
 'region',
 'country',
 'year',
 'month',
 'day',
 'day_of_week',
 'depth_flag',
 'mag_flag']

In [37]:
df_result6 = pd.read_sql(query6, engine)
print(df_result6)

   year  total_earthquakes
0  2026               9192
1  2025              22804
2  2024              18633
3  2023              20815
4  2022              20178
5  2021              21909


7. Month with highest number of earthquakes.

In [38]:
query7=''' SELECT month,COUNT(*) AS total_earthquakes
            FROM earthquakes
            GROUP BY month
            order by total_earthquakes desc'''

In [39]:
df_result7 = pd.read_sql(query7, engine)
print(df_result7)

        month  total_earthquakes
0       March              10890
1     January              10553
2       April              10282
3    February              10036
4      August               9848
5        July               9821
6         May               9669
7    December               9660
8        June               8390
9   September               8309
10    October               8140
11   November               7933


8. Day of week with most earthquakes.

In [40]:
query8=''' SELECT day_of_week, COUNT(*) AS total_earthquakes
            FROM earthquakes
            GROUP BY day_of_week
            ORDER BY total_earthquakes DESC'''

In [41]:
df_result8 = pd.read_sql(query8, engine)
print(df_result8)

  day_of_week  total_earthquakes
0     Tuesday              16392
1      Friday              16372
2      Sunday              16343
3   Wednesday              16247
4      Monday              16167
5    Saturday              16104
6    Thursday              15906


9. Count of earthquakes per hour of day.

In [42]:
query9='''SELECT EXTRACT(HOUR FROM time::TIMESTAMP) AS hour_of_day,
            COUNT(*) AS total_earthquakes
            FROM earthquakes
            GROUP BY hour_of_day
            ORDER BY hour_of_day ASC'''

In [43]:
df_result9 = pd.read_sql(query9, engine)
print(df_result9)

    hour_of_day  total_earthquakes
0           0.0               4717
1           1.0               4977
2           2.0               4827
3           3.0               5046
4           4.0               4994
5           5.0               4656
6           6.0               4577
7           7.0               4725
8           8.0               4716
9           9.0               4679
10         10.0               4550
11         11.0               4470
12         12.0               4432
13         13.0               4680
14         14.0               4532
15         15.0               4666
16         16.0               4573
17         17.0               4702
18         18.0               4757
19         19.0               4801
20         20.0               4816
21         21.0               5020
22         22.0               4907
23         23.0               4711


10.   Most active reporting network (net).

In [44]:
query10='''SELECT net, COUNT(*) AS total_earthquakes
             FROM earthquakes
             GROUP BY net
             ORDER BY total_earthquakes DESC
             LIMIT 10'''

In [45]:
df_result10 = pd.read_sql(query10, engine)
print(df_result10)

  net  total_earthquakes
0  us              99347
1  pr               6227
2  ak               3556
3  tx               1009
4  nc                929
5  ci                874
6  hv                859
7  nn                444
8  ok                 92
9  uu                 81


Casualties & Economic Loss

11.  Top 5 places with highest casualties.

In [46]:
query11='''SELECT country, sig,felt, mag, depth_km
             FROM earthquakes
             ORDER BY felt DESC
             LIMIT 5'''
             
# felt no of people who felt the earthquake

In [47]:
df_result11 = pd.read_sql(query11, engine)
print(df_result11)

      country   sig      felt   mag  depth_km
0  New Jersey   894  184666.0  4.80     2.624
1   Tennessee   739   45934.0  4.10    25.360
2          CA  1098   42594.0  5.21    14.290
3       Maine   652   42440.0  3.80    10.646
4          CA   904   33073.0  4.59    10.430


12.  Total estimated economic loss per continent.

13.  Average economic loss by alert level.

In [48]:
query13="""SELECT alert, COUNT(*) AS count
        FROM earthquakes
        GROUP BY alert"""

In [49]:
df_result13 = pd.read_sql(query13, engine)
print(df_result13)

    alert   count
0  yellow     130
1   green  113352
2     red      24
3  orange      25


Event Type & Quality Metrics

14.  Count of reviewed vs automatic earthquakes (status).

In [50]:
query14=''' select status, count(*) as total_earthquakes
            from earthquakes
            group by status'''

In [51]:
df_result14 = pd.read_sql(query14, engine)
print(df_result14)

      status  total_earthquakes
0   reviewed             113363
1  automatic                168


15.  Count by earthquake type (type).

In [52]:
query15='''select type, count(*) as total_earthquakes
            from earthquakes
            group by type'''

In [53]:
df_result15 = pd.read_sql(query15, engine)
print(df_result15)

                     type  total_earthquakes
0               ice quake                 13
1              earthquake             112672
2           mine collapse                  3
3        mining explosion                818
4               landslide                  4
5  experimental explosion                  3
6             other event                  5
7       volcanic eruption                 12
8               explosion                  1


16.  Number of earthquakes by data type (types).

In [54]:
query16='''select types, count(*) as total_earthquakes
            from earthquakes
            group by types'''

In [55]:
df_result16 = pd.read_sql(query16, engine)
print(df_result16)

                                                 types  total_earthquakes
0    dyfi,generallink,groundfailure,impactlink,loss...                  1
1    associate,disassociate,dyfi,groundfailure,impa...                  1
2    dyfi,internalmomenttensor,internalorigin,inter...                  1
3          focalmechanism,origin,phasedata,scitechlink                  1
4    dyfi,earthquakename,impacttext,internalmomentt...                  1
..                                                 ...                ...
571  generaltext,internalmomenttensor,momenttensor,...                  2
572           earthquakename,origin,phasedata,shakemap                 36
573  dyfi,focalmechanism,momenttensor,nearbycities,...                 56
574  dyfi,earthquakename,groundfailure,impacttext,l...                  1
575  associate,dyfi,focalmechanism,losspager,moment...                  1

[576 rows x 2 columns]


17.  Average RMS and gap per continent.

In [56]:
print("Average rms and gap per continent was not available in the dataset, so we cannot calculate it.")

Average rms and gap per continent was not available in the dataset, so we cannot calculate it.


18.  Events with high station coverage (nst > threshold).

In [57]:
query18='''SELECT place, mag, nst
FROM earthquakes
WHERE nst IS NOT NULL
ORDER BY nst DESC
LIMIT 10'''

In [58]:
df_result18 = pd.read_sql(query18, engine)
print(df_result18)

                                               place  mag    nst
0                          11 km W of Anamizu, Japan  5.4  619.0
1                  120 km ENE of Ozernovskiy, Russia  5.0  566.0
2                        111 km N of Yakutat, Alaska  5.7  516.0
3                                   Macquarie Island  6.8  475.0
4  49 km WNW of San Antonio de los Cobres, Argentina  5.5  466.0
5                                North Pacific Ocean  6.0  452.0
6                                   Kermadec Islands  5.1  444.0
7    34 km NE of Olonkinbyen, Svalbard and Jan Mayen  6.5  437.0
8                 280 km ESE of Attu Station, Alaska  5.9  437.0
9                    24 km NNW of Herāt, Afghanistan  6.3  423.0


Tsunamis & Alerts

19.  Number of tsunamis triggered per year.

In [59]:
query19=''' select year, tsunami, count(*) as total_earthquakes
            from earthquakes
            group by year, tsunami
            order by year desc'''

In [60]:
df_result19 = pd.read_sql(query19, engine)
print(df_result19)

    year  tsunami  total_earthquakes
0   2026        0               9158
1   2026        1                 34
2   2025        1                142
3   2025        0              22662
4   2024        0              18519
5   2024        1                114
6   2023        0              20696
7   2023        1                119
8   2022        1                136
9   2022        0              20042
10  2021        0              21795
11  2021        1                114


20.  Count earthquakes by alert levels (red, orange, etc.).

In [61]:
query20=''' select alert, count(*) as total_earthquakes
            from earthquakes
            group by alert
            order by alert desc'''

In [62]:
df_result20 = pd.read_sql(query20, engine)
print(df_result20)

    alert  total_earthquakes
0  yellow                130
1     red                 24
2  orange                 25
3   green             113352


Seismic Pattern & Trends Analysis.

21.Find the top 5 countries with the highest average magnitude of earthquakes in the past 5 years    

In [63]:
query21=''' select avg(mag) as average_magnitude, country from earthquakes
            group by country
            order by average_magnitude desc
            limit 5'''

In [64]:
df_result21 = pd.read_sql(query21, engine)
print(df_result21)

   average_magnitude                        country
0               8.10    2021 South Sandwich Islands
1               7.65                  Kahramanmaras
2               7.50    2025 Southern Drake Passage
3               7.40             2025 Drake Passage
4               7.10  2025 Southern Tibetan Plateau


22.Find countries that have experienced both shallow and deep earthquakes within the same month.

In [65]:
query22='''SELECT  year,month,
             COUNT(CASE WHEN depth_flag = 'Shallow' THEN 1 END) AS shallow_count,
             COUNT(CASE WHEN depth_flag = 'Deep' THEN 1 END) AS deep_count
             FROM earthquakes
             GROUP BY  year,month
             HAVING
                COUNT(CASE WHEN depth_flag = 'Shallow' THEN 1 END) > 0
                AND
                COUNT(CASE WHEN depth_flag = 'Deep' THEN 1 END) > 0
             ORDER BY year ASc'''

In [66]:
df_result22 = pd.read_sql(query22, engine)
print(df_result22)

    year     month  shallow_count  deep_count
0   2021     April           1160         416
1   2021    August           2257         421
2   2021  December           1480         434
3   2021  February           1446         369
4   2021   January           1227         485
..   ...       ...            ...         ...
61  2026  February           1106         415
62  2026   January           1235         434
63  2026      June            509         164
64  2026     March           1377         461
65  2026       May           1116         401

[66 rows x 4 columns]


23.Compute the year-over-year growth rate in the total number of earthquakes globally.

In [67]:
query23='''SELECT
    year,
    COUNT(*) AS total_earthquakes
FROM earthquakes
GROUP BY year
ORDER BY year;'''

In [68]:
df_result23 = pd.read_sql(query23, engine)
print(df_result23)

   year  total_earthquakes
0  2021              21909
1  2022              20178
2  2023              20815
3  2024              18633
4  2025              22804
5  2026               9192


24. List the 3 most seismically active regions by combining both frequency and average magnitude.

In [69]:
query24=''' SELECT
    place,
    COUNT(*) AS frequency,
    ROUND(AVG(mag)::numeric,2) AS avg_magnitude
FROM earthquakes
GROUP BY place
ORDER BY frequency DESC, avg_magnitude DESC
LIMIT 10'''

In [70]:
df_result24 = pd.read_sql(query24, engine)
print(df_result24)

                                   place  frequency  avg_magnitude
0                 South Sandwich Islands       3122           4.68
1              south of the Fiji Islands       1992           4.44
2                       Kermadec Islands       1911           4.65
3                                   Fiji       1298           4.37
4       southeast of the Loyalty Islands       1072           4.70
5                     Izu Islands, Japan       1036           4.53
6          Kermadec Islands, New Zealand        897           4.67
7  Rat Islands, Aleutian Islands, Alaska        837           3.50
8          south of the Kermadec Islands        770           4.57
9                        Reykjanes Ridge        620           4.59


Depth, Location & Distance-Based  Analysis.

25. For each country, calculate the average depth of earthquakes within ±5° latitude range of the equator.

In [71]:
query25=''' select country,avg(depth_km) as average_depth,count(*) as total_earthquakes from earthquakes
where latitude >= -5 and latitude <= 5 group by country 
order by average_depth desc'''

In [72]:
df_result25 = pd.read_sql(query25, engine)
print(df_result25)

                                   country  average_depth  total_earthquakes
0                              Celebes Sea     452.787273                 11
1                                Banda Sea     339.384000                 11
2                              Philippines     119.527538                519
3                             Bismarck Sea     106.018333                  6
4                         Papua New Guinea      71.647976               1170
5                                     Peru      63.975117                 94
6                                Indonesia      62.357706               5744
7                                  Ecuador      60.818469                324
8                      Peru-Ecuador border      59.683000                  5
9                            northern Peru      55.071000                  2
10                             Molucca Sea      51.642222                 72
11                                Colombia      42.241008                120


26. Identify countries having the highest ratio of shallow to deep earthquakes.

In [73]:
query26='''SELECT country,
             COUNT(CASE WHEN depth_flag = 'Shallow' THEN 1 END) AS shallow_count,
             COUNT(CASE WHEN depth_flag = 'Deep' THEN 1 END) AS deep_count,
             ROUND(
                COUNT(CASE WHEN depth_flag = 'Shallow' THEN 1 END)::NUMERIC /
                NULLIF(COUNT(CASE WHEN depth_flag = 'Deep' THEN 1 END), 0)
             , 2) AS shallow_to_deep_ratio
             FROM earthquakes
             GROUP BY country
             HAVING COUNT(CASE WHEN depth_flag = 'Deep' THEN 1 END) > 0
             ORDER BY shallow_to_deep_ratio DESC
             LIMIT 10'''

In [74]:
df_result26 = pd.read_sql(query26, engine)
print(df_result26)

               country  shallow_count  deep_count  shallow_to_deep_ratio
0                 Iran            963           3                 321.00
1               Canada            288           1                 288.00
2               Panama            220           2                 110.00
3                Nepal            150           2                  75.00
4                China           1369          19                  72.05
5              Morocco             65           1                  65.00
6               Turkey            957          16                  59.81
7             Pakistan            231           5                  46.20
8               Cyprus             39           1                  39.00
9  Antigua and Barbuda             33           1                  33.00



27. Find the average magnitude difference between earthquakes with tsunami alerts and those without.

In [75]:
query27='''SELECT 
             ROUND(AVG(CASE WHEN tsunami = 1 THEN mag END)::NUMERIC, 2) AS avg_mag_with_tsunami,
             ROUND(AVG(CASE WHEN tsunami = 0 THEN mag END)::NUMERIC, 2) AS avg_mag_without_tsunami,
             ROUND((
                AVG(CASE WHEN tsunami = 1 THEN mag END) -
                AVG(CASE WHEN tsunami = 0 THEN mag END)
             )::NUMERIC, 2) AS magnitude_difference
             FROM earthquakes'''

In [76]:
df_result27 = pd.read_sql(query27, engine)
print(df_result27)

   avg_mag_with_tsunami  avg_mag_without_tsunami  magnitude_difference
0                  5.42                     4.25                  1.17


28. Using the gap and rms columns, identify events with the lowest data reliability (highest average error margins).

In [77]:
query28='''SELECT
    place,
    gap,
    rms,
    mag
FROM earthquakes
WHERE gap IS NOT NULL
AND rms IS NOT NULL
ORDER BY gap DESC, rms DESC
LIMIT 10'''

In [78]:
df_result28 = pd.read_sql(query28, engine)
print(df_result28)

                                               place     gap     rms   mag
0         95 km ENE of Cruz Bay, U.S. Virgin Islands  359.00  0.1400  3.31
1                          60 km NE of Valmy, Nevada  358.18  0.1601  3.00
2         77 km NW of Sandy Ground Village, Anguilla  358.00  0.1600  3.58
3        86 km WNW of Sandy Ground Village, Anguilla  356.00  0.2200  3.17
4        Andreanof Islands, Aleutian Islands, Alaska  355.00  0.3400  3.30
5         59 km ENE of Cruz Bay, U.S. Virgin Islands  353.00  0.6100  3.63
6      53 km SSW of Boca de Yuma, Dominican Republic  352.00  0.1800  3.13
7           78 km E of Cruz Bay, U.S. Virgin Islands  352.00  0.0700  3.48
8  87 km WNW of The Bottom, Bonaire, Saint Eustat...  351.00  0.4400  3.53
9          58 km NE of Cruz Bay, U.S. Virgin Islands  351.00  0.1700  3.13


29. Find pairs of consecutive earthquakes (by time) that occurred within 50 km of each other and within 1 hour.

30. Determine the regions with the highest frequency of deep-focus earthquakes (depth > 300 km).

In [82]:
query30='''SELECT
    place,
    COUNT(*) AS total_earthquakes,
    AVG(depth_km) AS avg_depth,
    MAX(depth_km) AS max_depth,
    AVG(mag) AS avg_magnitude
FROM earthquakes
WHERE depth_km > 300
GROUP BY place
ORDER BY total_earthquakes DESC;'''

In [83]:
df_result30 = pd.read_sql(query30, engine)
print(df_result30)

                                              place  total_earthquakes  \
0                         south of the Fiji Islands               1672   
1                                              Fiji               1250   
2                                  Kermadec Islands                162   
3                              Bonin Islands, Japan                135   
4                                Izu Islands, Japan                111   
...                                             ...                ...   
1846                   268 km S of Ambon, Indonesia                  1   
1847          220 km WNW of Severo-Kuril’sk, Russia                  1   
1848                173 km N of Baukau, Timor Leste                  1   
1849                      57 km NNW of Malfa, Italy                  1   
1850  39 km WNW of Saipan, Northern Mariana Islands                  1   

       avg_depth  max_depth  avg_magnitude  
0     522.855011    664.740       4.415251  
1     556.758939    6

In [81]:
skip_queries = [4, 12, 13, 17, 29]

for i in range(1, 31):
    if i in skip_queries:
        continue

    var_name = f"query{i}"

    if var_name in globals():
        print(f"\n{'='*80}")
        print(f"QUERY {i}")
        print(globals()[var_name])
        print(f"{'='*80}")


QUERY 3
SELECT depth_km,place,mag FROM earthquakes
        Where depth_km <50 AND mag > 7.5
        ORDER BY mag DESC

QUERY 5
 SELECT "magType", AVG(mag) AS avg_magnitude
            FROM earthquakes
            GROUP BY "magType"
            ORDER BY avg_magnitude DESC

QUERY 6
 SELECT year,COUNT(*) AS total_earthquakes
            FROM earthquakes
            GROUP BY year
            order by year desc

QUERY 7
 SELECT month,COUNT(*) AS total_earthquakes
            FROM earthquakes
            GROUP BY month
            order by total_earthquakes desc

QUERY 8
 SELECT day_of_week, COUNT(*) AS total_earthquakes
            FROM earthquakes
            GROUP BY day_of_week
            ORDER BY total_earthquakes DESC

QUERY 9
SELECT EXTRACT(HOUR FROM time::TIMESTAMP) AS hour_of_day,
            COUNT(*) AS total_earthquakes
            FROM earthquakes
            GROUP BY hour_of_day
            ORDER BY hour_of_day ASC

QUERY 10
SELECT net, COUNT(*) AS total_earthquakes
          